# Session 2: Prompt Engineering for Agentic Behaviors (Instructor Notebook — Full Solutions)

## Learning Objectives

By the end of this session, students will be able to:

1. **Craft effective system prompts** that define agent personas, capabilities, and constraints
2. **Implement Chain-of-Thought (CoT) and ReAct prompting** patterns to improve reasoning quality
3. **Generate structured outputs** (JSON mode) for reliable downstream processing
4. **Manage multi-turn conversations** with context accumulation and summarization strategies

In [ ]:
# Imports and Setup
import openai
import json
import os

# Ensure your OPENAI_API_KEY is set in your environment variables
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

---
## Demos

Walk through these demos with the students. Run each cell and discuss the outputs.

### Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.  
**Few-shot** prompting provides a handful of examples so the model can learn the expected format and behavior.

We will compare both approaches on a sentiment classification task.

**Instructor Note:** Emphasize that few-shot prompting is one of the simplest yet most effective techniques for controlling output format. Point out the difference in verbosity between the two responses.

In [ ]:
# Demo 1: Zero-Shot vs. Few-Shot Prompting

# Zero-shot: no examples provided
zero_shot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Classify the sentiment of this review as positive, negative, or neutral: 'The product works fine but the delivery was terrible.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

print("\n" + "="*60 + "\n")

# Few-shot: provide labeled examples first
few_shot_messages = [
    {"role": "system", "content": "You are a sentiment classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Review: 'Absolutely loved this product!'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Review: 'Worst purchase ever, total waste of money.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Review: 'The product works fine but the delivery was terrible.'"}
]
few_shot_response = client.chat.completions.create(model="gpt-4o-mini", messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

**Observation:** Notice how the zero-shot response may include extra explanation, while the few-shot response follows the exact format demonstrated in the examples.

### Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning process step by step, which improves accuracy on math, logic, and multi-step problems.

**Instructor Note:** The key insight here is that LLMs perform better when they "think out loud." This is because intermediate tokens act as a scratchpad for computation.

In [ ]:
# Demo 2: Chain-of-Thought Prompting

problem = "A store has 45 apples. They sell 60% in the morning and half of the remainder in the afternoon. How many are left?"

# Without CoT
response_no_cot = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

print("\n" + "="*60 + "\n")

# With explicit CoT
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many were sold in the morning\n2. Then, find the remainder\n3. Calculate how many were sold in the afternoon\n4. Find the final count"
response_cot = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

**Observation:** The CoT version explicitly walks through each calculation, making it easier to verify correctness and debug any errors in reasoning.

### Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different agent personas that approach the same question from different perspectives. This is foundational for building specialized agents.

**Instructor Note:** Discuss how multi-agent systems leverage this -- each agent can be a different persona working on the same problem from a different angle.

In [ ]:
# Demo 3: Role-Based Prompting for Agentic Personas

personas = {
    "Data Analyst": "You are a senior data analyst. You approach every question with data-driven thinking, ask for metrics, and suggest analytical frameworks.",
    "Security Expert": "You are a cybersecurity expert. You evaluate everything through the lens of security risks, vulnerabilities, and compliance requirements.",
    "Product Manager": "You are a product manager. You think about user needs, business value, prioritization, and stakeholder communication."
}
question = "We want to add a new feature to our web application that allows users to upload files."

for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=200
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")

**Observation:** Each persona focuses on different aspects of the same question -- data metrics, security risks, or user/business impact. This demonstrates how system prompts shape agent behavior.

### Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems, we often need the LLM output to be machine-readable. JSON mode ensures the response is valid JSON, making downstream processing reliable.

**Instructor Note:** Highlight that `response_format={"type": "json_object"}` is an API-level guarantee. Without it, the model might produce mostly-valid JSON but occasionally break format.

In [ ]:
# Demo 4: Structured Output Generation (JSON Mode)

text = "John Smith is a 35-year-old software engineer from San Francisco. He has 10 years of experience and specializes in Python and machine learning."

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract the person's information and return it as a JSON object with keys: name, age, occupation, location, experience_years, skills (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nName: {parsed.get('name')}")
print(f"Skills: {', '.join(parsed.get('skills', []))}")

**Observation:** With `response_format={"type": "json_object"}`, the output is guaranteed to be valid JSON, which we can reliably parse and use in code.

### Demo 5: Multi-Turn Conversation Management

Agents need to maintain context across multiple exchanges. This demo shows how to build a conversation manager that accumulates context and tracks history.

**Instructor Note:** Discuss the trade-off between keeping full history (better context) and truncating/summarizing (staying within token limits). In production, you need a strategy for managing context window size.

In [ ]:
# Demo 5: Multi-Turn Conversation Management

class ConversationManager:
    def __init__(self, system_prompt, model="gpt-4o-mini"):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=300
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

# Demo the conversation
conv = ConversationManager("You are a helpful travel planning assistant.")

print("User: I want to plan a trip to Japan.")
print("Assistant:", conv.send("I want to plan a trip to Japan."))

print("\n" + "-"*50)
print("\nUser: What's the best time to visit?")
print("Assistant:", conv.send("What's the best time to visit?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize what we've discussed so far?")
print("Assistant:", conv.send("Can you summarize what we've discussed so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

**Observation:** The assistant remembers context from earlier turns -- it knows we are discussing Japan without being reminded. The conversation history grows with each exchange.

---
## Tasks (Full Solutions)

Below are the complete, working solutions for all four tasks. Use these as a reference when guiding students.

### Task 1: Design Effective System Prompts for a Research Agent

**Goal:** Create a comprehensive system prompt that defines a Research Agent's behavior, output format, and constraints.

**Key Teaching Points:**
- A good system prompt has three sections: role definition, output format specification, and behavioral guidelines.
- Being specific about the output format (section headers, structure) produces more consistent results.
- Including meta-instructions like "state your confidence level" makes the agent more trustworthy.

In [ ]:
# Task 1 Solution: Design Effective System Prompts for a Research Agent

research_agent_prompt = """You are a Research Agent specializing in analyzing complex topics and providing structured, evidence-based responses.

## Your Approach:
1. Break down the research question into sub-questions
2. Analyze each sub-question systematically
3. Synthesize findings into a coherent response

## Output Format:
Always structure your response as:
- **Summary**: 2-3 sentence overview
- **Key Findings**: Bullet points of main discoveries
- **Analysis**: Detailed exploration with reasoning
- **Confidence Level**: High/Medium/Low with justification
- **Limitations**: What you're uncertain about

## Guidelines:
- If uncertain, explicitly state your confidence level
- Distinguish between facts and inferences
- Provide balanced perspectives on controversial topics"""

def ask_research_agent(question):
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": research_agent_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=800
    )
    result = response.choices[0].message.content
    print(result)
    return result

ask_research_agent("What are the latest trends in renewable energy storage?")

### Task 2: Implement ReAct-Style Prompting (Reason + Act)

**Goal:** Implement the ReAct pattern where the model interleaves reasoning (Thought) with actions (Action) and observations (Observation).

**Key Teaching Points:**
- ReAct is a foundational pattern for agentic AI -- it mirrors how real agents interact with tools.
- In this demo, the LLM simulates both the reasoning and the tool results. In a real system, Actions would trigger actual API calls.
- The explicit format (Thought/Action/Observation) makes the reasoning chain transparent and debuggable.

In [ ]:
# Task 2 Solution: Implement ReAct-Style Prompting (Reason + Act)

def react_agent(question):
    react_prompt = """You are a ReAct agent that solves problems by interleaving reasoning and actions.

Available Actions:
- Search[query]: Search for factual information
- Calculate[expression]: Perform mathematical calculations
- Lookup[term]: Look up a specific term or definition

Follow this EXACT format for each step:
Thought: [Your reasoning about what to do next]
Action: [ActionName][input]
Observation: [What you found/result]

Continue until you have enough information, then provide:
Final Answer: [Your complete answer]

Always show your complete reasoning chain."""

    client = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": react_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=500
    )
    result = response.choices[0].message.content
    print(result)
    return result

react_agent("What is the population of the capital of France?")

### Task 3: Create a Reusable Prompt Template Library

**Goal:** Build a `PromptTemplate` class that manages parameterized prompts with validation.

**Key Teaching Points:**
- Template libraries reduce duplication and enforce consistency across an application.
- Validation prevents runtime errors from missing variables.
- This pattern is used in frameworks like LangChain (`PromptTemplate`) and is an industry best practice.

In [ ]:
# Task 3 Solution: Create a Reusable Prompt Template Library

import re

class PromptTemplate:
    def __init__(self, name, template, description=""):
        self.name = name
        self.template = template
        self.description = description
        self.variables = re.findall(r'\{(\w+)\}', template)
    
    def format(self, **kwargs):
        self.validate(**kwargs)
        return self.template.format(**kwargs)
    
    def validate(self, **kwargs):
        missing = [v for v in self.variables if v not in kwargs]
        if missing:
            raise ValueError(f"Missing required variables: {missing}")
        return True

# Template library
summarize_template = PromptTemplate(
    name="summarize",
    template="Summarize the following {content_type} in {length} sentences:\n\n{text}",
    description="Summarizes content to a specified length"
)

classify_template = PromptTemplate(
    name="classify",
    template="Classify the following text into one of these categories: {categories}.\n\nText: {text}\n\nRespond with only the category name.",
    description="Classifies text into predefined categories"
)

extract_template = PromptTemplate(
    name="extract",
    template="Extract the following fields from the text: {fields}.\n\nText: {text}\n\nReturn the result as a JSON object.",
    description="Extracts structured fields from text"
)

# Demo usage
prompt = summarize_template.format(
    content_type="article",
    length="3",
    text="Artificial intelligence has transformed the technology landscape in recent years. From natural language processing to computer vision, AI systems are now capable of performing tasks that were once thought to be exclusively human. The rapid advancement has raised both excitement about potential benefits and concerns about ethical implications."
)
print(f"Generated prompt:\n{prompt}")
print(f"\nTemplate variables: {summarize_template.variables}")

# Show validation error
print("\n--- Testing validation ---")
try:
    summarize_template.format(content_type="article")
except ValueError as e:
    print(f"Validation error (expected): {e}")

### Task 4: Build a Multi-Step Reasoning Agent with Tool Descriptions

**Goal:** Build a complete agentic loop where the LLM decides which tool to call, receives simulated results, and iterates until it has a final answer.

**Key Teaching Points:**
- This is the core pattern of tool-using agents: perceive -> reason -> act -> observe -> repeat.
- JSON mode is essential here so the agent's tool calls are machine-parseable.
- The `simulate_tool` method stands in for real tool integrations -- in production, these would be API calls.
- The loop has a max-step limit to prevent infinite loops (an important safety measure).

In [ ]:
# Task 4 Solution: Build a Multi-Step Reasoning Agent with Tool Descriptions

tools = [
    {"name": "calculator", "description": "Performs mathematical calculations", "parameters": "expression (string)"},
    {"name": "search", "description": "Searches for factual information", "parameters": "query (string)"},
    {"name": "weather", "description": "Gets current weather for a location", "parameters": "city (string)"}
]

class ToolAgent:
    def __init__(self):
        self.client = openai.OpenAI()
        tool_descriptions = "\n".join([f"- {t['name']}: {t['description']} (params: {t['parameters']})" for t in tools])
        self.system_prompt = f"""You are an AI agent with access to the following tools:

{tool_descriptions}

When you need to use a tool, respond with JSON:
{{"thought": "your reasoning", "tool": "tool_name", "input": "tool_input"}}

When you have enough information to answer, respond with JSON:
{{"thought": "final reasoning", "tool": "final_answer", "input": "your complete answer"}}

Process one step at a time."""
    
    def simulate_tool(self, tool_name, tool_input):
        simulated = {
            "calculator": f"Result: {tool_input} = [calculated value]",
            "search": f"Search results for '{tool_input}': [relevant information found]",
            "weather": f"Weather in {tool_input}: 72\u00b0F, partly cloudy"
        }
        return simulated.get(tool_name, "Tool not found")
    
    def process(self, question):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": question}
        ]
        
        for step in range(5):  # Max 5 steps
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                response_format={"type": "json_object"},
                max_tokens=300
            )
            result = json.loads(response.choices[0].message.content)
            print(f"\nStep {step + 1}:")
            print(f"  Thought: {result.get('thought', 'N/A')}")
            print(f"  Tool: {result.get('tool', 'N/A')}")
            print(f"  Input: {result.get('input', 'N/A')}")
            
            if result.get("tool") == "final_answer":
                print(f"\nFinal Answer: {result['input']}")
                return result["input"]
            
            observation = self.simulate_tool(result["tool"], result["input"])
            print(f"  Observation: {observation}")
            messages.append({"role": "assistant", "content": response.choices[0].message.content})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        
        return "Max steps reached"

agent = ToolAgent()
agent.process("What's 15% tip on a $85 dinner, and what's the weather like in New York?")

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.